<div style="max-width:300px; float: left; margin-right: 1em">

![](Figures/fcfm_das.svg)

</div>
<span style="color: red;">Departamento de Astronomía, Universidad de Chile</span> <br>
Facultad de Ciencias Físicas y Matemáticas <br>
Universidad de Chile <br>
AS4501 - Astroinformatica <br>
Otoño 2026 <br>
Profesor: Francisco Förster Burón <br>
<strong>Profesores Auxilares y Autores: Benjamin Carrera y Steve Jurado</strong> <br>

> This notebook is created based on the notes from 
> - ([@fforster](https://github.com/fforster)) Francisco Förster: - **Main Notes 2026/01**   
>   - https://github.com/fforster/AS4501/tree/main
> - ([@thevalentino](https://github.com/thevalentino)) Valentino Gonzales:
>   - https://github.com/thevalentino/AS450-astroinformatica
> - ([@cefuente](https://github.com/cefuente)) Cesar Fuentes
>   - https://github.com/cefuente/astroinformatica
> 
> and previour teachers assistants
> - ([@m-fuentealba](https://github.com/m-fuentealba)) Melissa Fuentealba 
> - ([@jvines](https://github.com/jvines)) José Vines
> - ([@PauCaBu](https://github.com/PauCaBu)) Paula Cáceres Burgos
> - ([@JavieraTGrey](https://github.com/JavieraTGrey)) Javiera Toro Grey

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached markupsafe-3.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.7 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 5.0 MB/s eta 0:00:0000:01m00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 9.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 7.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 13.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 9.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 9.5 MB/s eta 0:00:00:00:0100:01
  

In [7]:
import io
import requests
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Astronomy 
from astroquery.sdss import SDSS

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

rc_params = {
    # --- Fonts and text ---
    'mathtext.fontset': 'cm',     # Set default mathtext font
    'font.family': 'STIXGeneral', # Set default font family
    
    # --- Figure and axes ---
    'font.size': 12,              # Set default font size
    'axes.labelsize': 16,         # Set default axes label size
    'axes.titlesize': 16,         # Set default axes title size
    'xtick.labelsize': 14,        # Set default axes label size
    'ytick.labelsize': 14,        # Set default axes label size
    'legend.fontsize': 14,        # Set default legend font size
    
    # --- Configuration of ticks ---
    'xtick.direction': 'in',      # Set default xtickdirecion
    'ytick.direction': 'in',      # Set default ytickdirecion
    'xtick.minor.visible': True,  # visibility of minor ticks on x-axis
    'ytick.minor.visible': True,  # visibility of minor ticks on y-axis


    'grid.linestyle': ':',        # Set grid linestyle
    'grid.alpha': 0.6,            # Set grid transparency
    
    # --- Figure size ---
    'figure.figsize': (8, 6),     # Ideal proportion for one MNRAS column
}
plt.rcParams.update(rc_params)

In [48]:
%%capture
try: 
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import lightning.pytorch as L
except:
    !pip install torch lightning 
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader
    import lightning.pytorch as L

In [49]:
query = f"""
SELECT TOP {500}
    p.objid, p.u, p.g, p.r, p.i, p.z, s.z as spec_z
FROM PhotoObj AS p
JOIN SpecObj AS s ON s.bestobjid = p.objid
WHERE 
    p.clean = 1 AND 
    s.class = 'GALAXY' AND 
    s.zWarning = 0 AND
    p.r > 0 AND p.u > 0 -- Filtro básico para valores anómalos
"""
resultado = SDSS.query_sql(query)
df = resultado.to_pandas()

In [50]:
df.shape, df.columns, df.head()

((500, 7),
 Index(['objid', 'u', 'g', 'r', 'i', 'z', 'spec_z'], dtype='str'),
                  objid         u         g  ...         i         z    spec_z
 0  1237646379930419437  24.37175  21.57891  ...  18.60767  17.95021  0.000036
 1  1237646585565413466  21.09426  18.00359  ...  15.18166  14.47319  0.000239
 2  1237646585567118892  23.61513  21.95656  ...  18.98824  18.28613 -0.000011
 3  1237646586103924417  23.89194  22.37032  ...  19.31911  18.54503 -0.000054
 4  1237646586638566303  22.47528  21.26304  ...  19.84324  19.52212  0.000218
 
 [5 rows x 7 columns])

In [51]:
class GalaxyTransformer(nn.Module):
    def __init__(self, num_features=5, d_model=64, nhead=4, num_layers=3, dropout=0.1):
        super().__init__()

        self.feature_embeddings = nn.ModuleList([
            nn.Linear(1, d_model) for _ in range(num_features)
        ])

        encoder_layer = nn.TransformerEncoderLayer(
            d_model     = d_model,
            nhead       = nhead,
            dropout     = dropout,
            batch_first = True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)

        self.regressor = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model //2, 1) # Salida el redshift estimado
        )

    def forward(self,x):
        # x shape: (batch_size, num_features)
        batch_size   = x.size(0)
        num_features = x.size(1)

        # Generar embeddings para cada característica
        # tokens shape final: (batch_size, num_features, d_model)
        tokens = torch.stack([
            self.feature_embeddings[i](x[:, i].unnsqueeze(-1))
            for i in range()
        ])

        transformed = sefl.transformer_encoder(tokens)
        pooled = transformed.mean(dim=1)

        out = self.regressor(pooled)
        return out.squeze(-1)

In [52]:
class GalaxyRedshiftLightning(L.LightningModule):
    def __init__(self, lr=1e-3, **kwargs):
        super().__init__()
        self.save_hyperparameters()
        self.model = GalaxyTransformer(**kwargs)
        self.criterion = nn.MSELoss() # Error Cuadrático Medio para regresión
        self.lr = lr

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        self.log('val_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        # AdamW es altamente recomendado para Transformers
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.lr, weight_decay=1e-4)
        
        # Un scheduler para reducir la tasa de aprendizaje si nos estancamos
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "monitor": "val_loss"
            }
        }

In [53]:
def preparar_dataloaders(df, batch_size=256):
    """
    Toma el DataFrame descargado del SDSS, escala las características 
    y devuelve los DataLoaders de PyTorch listos para entrenar.
    """
    print("Iniciando preprocesamiento de datos astronómicos...")
    
    # 1. Separar características (X: fotometría) y objetivo (y: redshift)
    X = df[['u', 'g', 'r', 'i', 'z']].values
    y = df['spec_z'].values
    
    # 2. División de conjuntos: 70% Train, 15% Val, 15% Test
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1764, random_state=42)
    
    # 3. Estandarización (z-score normalization)
    # ¡CRÍTICO!: El scaler se ajusta (fit) SOLO con los datos de entrenamiento
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # 4. Conversión a Tensores de PyTorch
    # Es importante usar float32 para la compatibilidad con las redes neuronales
    def to_tensor(features, targets):
        return TensorDataset(
            torch.tensor(features, dtype=torch.float32),
            torch.tensor(targets, dtype=torch.float32)
        )
        
    train_dataset = to_tensor(X_train_scaled, y_train)
    val_dataset = to_tensor(X_val_scaled, y_val)
    test_dataset = to_tensor(X_test_scaled, y_test)
    
    # 5. Creación de los DataLoaders
    # shuffle=True en train rompe posibles correlaciones temporales/espaciales en el catálogo
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    print(f"DataLoaders listos. Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
    return train_loader, val_loader, test_loader, scaler

In [54]:
train_loader, val_loader, test_loader, scaler = preparar_dataloaders(df)

Iniciando preprocesamiento de datos astronómicos...
DataLoaders listos. Train: 350 | Val: 75 | Test: 75


In [ ]:
def entrenar_modelo(train_loader, val_loader):
    # 1. Instanciamos el módulo Lightning que contiene nuestro GalaxyTransformer
    modelo = GalaxyRedshiftLightning(
        num_features=5, 
        d_model=64, 
        nhead=4, 
        num_layers=3, 
        dropout=0.1, 
        lr=1e-3
    )
    
    # 2. Definimos callbacks (Buenas prácticas de ingeniería)
    # EarlyStopping: detiene el entrenamiento si no hay mejora, evitando el overfitting
    early_stop_callback = EarlyStopping(
        monitor='val_loss',
        patience=10,
        mode='min'
    )
    
    # ModelCheckpoint: Guarda los pesos del modelo con el menor error de validación
    checkpoint_callback = ModelCheckpoint(
        monitor='val_loss',
        dirpath='modelos_guardados/',
        filename='transformer-galaxy-{epoch:02d}-{val_loss:.4f}',
        save_top_k=1,
        mode='min'
    )
    
    # 3. Configuramos el Trainer
    # accelerator='auto' detectará automáticamente si tienes GPU (CUDA/MPS) o CPU
    trainer = L.Trainer(
        max_epochs=5,
        accelerator='auto', 
        devices=1,
        callbacks=[early_stop_callback],# checkpoint_callback],
        enable_progress_bar=True
    )
    
    # 4. ¡Comenzamos el entrenamiento!
    print("Iniciando el entrenamiento del Transformer...")
    trainer.fit(model=modelo, train_dataloaders=train_loader, val_dataloaders=val_loader)
    
    return modelo, trainer

In [56]:
modelo_entrenado, trainer = entrenar_modelo(train_loader, val_loader)

NameError: name 'EarlyStopping' is not defined

In [57]:
def predecir_redshift(modelo, test_loader, trainer):
    """
    Ejecuta el modelo en el conjunto de prueba y extrae las predicciones
    para realizar la posterior evaluación astrofísica.
    """
    print("Realizando inferencia en el conjunto de prueba...")
    
    # Opción A: Usar el Trainer de Lightning (recomendado para grandes volúmenes)
    # Retorna una lista de lotes (batches) con las predicciones
    predicciones_lotes = trainer.predict(model=modelo, dataloaders=test_loader)
    
    # Concatenamos todos los tensores en uno solo (1D array de predicciones)
    z_phot = torch.cat(predicciones_lotes).squeeze().numpy()
    
    # Extraemos el redshift verdadero (z_spec) del test_loader para comparar
    z_spec = torch.cat([batch[1] for batch in test_loader]).numpy()
    
    # Cálculo rápido del sesgo y error (opcional pero ilustrativo)
    mse = ((z_phot - z_spec)**2).mean()
    delta_z = z_phot - z_spec
    outliers = (abs(delta_z) / (1 + z_spec) > 0.15).mean() * 100
    
    print("\n--- Resultados Astrofísicos Preliminares ---")
    print(f"MSE en Test: {mse:.4f}")
    print(f"Fracción de Outliers Catastróficos: {outliers:.2f}%")
    
    return z_phot, z_spec

In [58]:
z_phot_pred, z_spec_true = predecir_redshift(modelo_entrenado, test_loader, trainer)


NameError: name 'modelo_entrenado' is not defined